# Chapter 1

In [ ]:
!pip install -U spacy[cuda110,transformers,lookups]==3.0.3
!pip install -U spacy-lookups-data==1.0.0
!pip install cupy-cuda110==8.5.0
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 71.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 MB 8.9 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement cupy-cuda110==8.5.0 (from versions: none)
ERROR: No matching distribution found for cupy-cuda110==8.5.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 10.5 M

## Tokenization

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm") # kiçikölçülü, ingilis dilinə əsaslanan veb-modeli

In [ ]:
sentence = nlp.tokenizer("We live in Paris.")
print(f"The number of tokens: {len(sentence)}")

The number of tokens: 5


In [ ]:
for words in sentence:
  print(f"The token: {words}")

The token: We
The token: live
The token: in
The token: Paris
The token: .


In [ ]:
import pandas as pd
import os

cwd = os.getcwd()
cwd

'/content'

In [ ]:
# Jeopardy questions
# data = pd.read_csv(cwd+'/data/jeopardy_questions/jeopardy_questions.csv')
# data = pd.DataFrame(data=data)

In [ ]:
!kaggle datasets download tunguz/200000-jeopardy-questions

Dataset URL: https://www.kaggle.com/datasets/tunguz/200000-jeopardy-questions
License(s): other
100% 11.5M/11.5M [00:00<00:00, 41.5MB/s]



In [ ]:
!unzip /content/200000-jeopardy-questions.zip

Archive:  /content/200000-jeopardy-questions.zip
  inflating: JEOPARDY_CSV.csv        


In [ ]:
data = pd.read_csv('/content/JEOPARDY_CSV.csv')
data.columns = map(lambda x: x.lower().strip(), data.columns)

In [ ]:
# reduce the data size
data = data[:1000]

In [ ]:
data['question_tokens'] = data['question'].apply(lambda x: nlp(x))

In [ ]:
# first question as a sample
example_question = data.question[0]
example_question_tokens = data.question_tokens[0]

In [ ]:
example_question

"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory"

In [ ]:
for token in example_question_tokens:
  print(token)

For
the
last
8
years
of
his
life
,
Galileo
was
under
house
arrest
for
espousing
this
man
's
theory


## Part-of-speech (POS) tagging

In [ ]:
for token in example_question_tokens:
  print(token.text, token.pos_, spacy.explain(token.pos_))

For ADP adposition
the DET determiner
last ADJ adjective
8 NUM numeral
years NOUN noun
of ADP adposition
his PRON pronoun
life NOUN noun
, PUNCT punctuation
Galileo PROPN proper noun
was AUX auxiliary
under ADP adposition
house NOUN noun
arrest NOUN noun
for ADP adposition
espousing VERB verb
this DET determiner
man NOUN noun
's PART particle
theory NOUN noun


## Dependency Parsing tags

In [ ]:
for token in example_question_tokens:
  print(token.text, token.dep_, spacy.explain(token.dep_))

For prep prepositional modifier
the det determiner
last amod adjectival modifier
8 nummod numeric modifier
years pobj object of preposition
of prep prepositional modifier
his poss possession modifier
life pobj object of preposition
, punct punctuation
Galileo nsubj nominal subject
was ROOT root
under prep prepositional modifier
house compound compound
arrest pobj object of preposition
for prep prepositional modifier
espousing pcomp complement of preposition
this det determiner
man poss possession modifier
's case case marking
theory dobj direct object


In [ ]:
# visualize the dependency parse
from spacy import displacy

displacy.render(example_question_tokens, style='dep', jupyter=True, options={"distance": 120})

In [ ]:
print(dir(spacy))

['Any', 'Config', 'ConfigSchema', 'ConfigSchemaInit', 'ConfigSchemaNlp', 'ConfigSchemaPretrain', 'ConfigSchemaTraining', 'Dict', 'Errors', 'Example', 'Iterable', 'Language', 'Path', 'REGISTRY_POPULATED', 'Union', 'Vocab', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_rebuild_ns', '_schema', 'about', 'attrs', 'blank', 'cli', 'compat', 'displacy', 'errors', 'explain', 'git_info', 'glossary', 'info', 'kb', 'lang', 'language', 'lexeme', 'load', 'logger', 'lookups', 'matcher', 'ml', 'morphology', 'parts_of_speech', 'pipe_analysis', 'pipeline', 'populate_registry', 'prefer_gpu', 'registrations', 'registry', 'require_cpu', 'require_gpu', 'schemas', 'scorer', 'setup_default_warnings', 'strings', 'symbols', 'sys', 'tokenizer', 'tokens', 'training', 'ty', 'util', 'vectors', 'vocab']


## Chunking

In [ ]:
# without chunking
for token in nlp("My parents live in New York City."):
  print(token.text)

My
parents
live
in
New
York
City
.


In [ ]:
## with chunking
for chunk in nlp("My parents live in New York City.").noun_chunks:
  print(chunk.text)

My parents
New York City


## Lemmatization

In [ ]:
lemmatization = pd.DataFrame(data=[], columns=["original", "lemmatized"])

In [ ]:
i = 0
for token in example_question_tokens:
  lemmatization.loc[i, "original"] = token.text
  lemmatization.loc[i, "lemmatized"] = token.lemma_
  i += 1

In [ ]:
lemmatization

,original,lemmatized
0,For,for
1,the,the
2,last,last
3,8,8
4,years,year
5,of,of
6,his,his
7,life,life
8,",",","
9,Galileo,Galileo


## Named entity recognition (NER)

In [ ]:
example_sentence = "George Washington was an American political leader, \
military general, statesman, and Founding Father who served as the \
first president of the United States from 1789 to 1797.\n"

print(example_sentence)

George Washington was an American political leader, military general, statesman, and Founding Father who served as the first president of the United States from 1789 to 1797.



In [ ]:
doc = nlp(example_sentence)
doc.ents

(George Washington, American, first, the United States, 1789 to 1797)

In [ ]:
for token in doc.ents:
  print(token.text, token.start_char, token.end_char, token.label_)

George Washington 0 17 PERSON
American 25 33 NORP
first 119 124 ORDINAL
the United States 138 155 GPE
1789 to 1797 161 173 DATE


In [ ]:
displacy.render(doc, style='ent', jupyter=True, options={"distance": 120})

## Named entity linking (NEL)

In [ ]:
import requests

# define Google Knowledge Graph API result function
def returnGraphResult(query, key, entityType):
    if entityType == "PERSON":
        google = f"https://kgsearch.googleapis.com/v1/entities:search?query={query}&key={key}"
        resp = requests.get(google)
        url = resp.json()['itemListElement'][0]['result']['detailedDescription']['url']
        description = resp.json()['itemListElement'][0]['result']['detailedDescription']['article']
        return url, description
    else:
        return "no_match", "no_match"

In [ ]:
key = "YOUR_GOOGLE_KNOWLEDGE_GRAPH_API_KEY"

for token in doc.ents:
    if key != "YOUR_GOOGLE_KNOWLEDGE_GRAPH_API_KEY":
        try:
            url, description = returnGraphResult(token.text, key, token.label_)
            print(token.text, token.label_, url, description)
        except Exception as e:
            print(f"Could not fetch details for {token.text}: {e}")
    else:
        print(f"{token.text} ({token.label_}) - [Please set your API key to fetch Graph Results]")

George Washington (PERSON) - [Please set your API key to fetch Graph Results]
American (NORP) - [Please set your API key to fetch Graph Results]
first (ORDINAL) - [Please set your API key to fetch Graph Results]
the United States (GPE) - [Please set your API key to fetch Graph Results]
1789 to 1797 (DATE) - [Please set your API key to fetch Graph Results]


# Chapter 2

## Training with fastai

In [1]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

<div><progress max="144440600" value="144441344"></progress> 100.00% [144441344/144440600 00:10&lt;00:00]</div>

In [2]:
dls = TextDataLoaders.from_folder(path, valid='test')

In [3]:
dls.show_batch()

,text,category
0,"xxbos xxmaj match 1 : xxmaj tag xxmaj team xxmaj table xxmaj match xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley vs xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley started things off with a xxmaj tag xxmaj team xxmaj table xxmaj match against xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit . xxmaj according to the rules of the match , both opponents have to go through tables in order to get the win . xxmaj benoit and xxmaj guerrero heated up early on by taking turns hammering first xxmaj spike and then xxmaj bubba xxmaj ray . a xxmaj german xxunk by xxmaj benoit to xxmaj bubba took the wind out of the xxmaj dudley brother . xxmaj spike tried to help his brother , but the referee restrained him while xxmaj benoit and xxmaj guerrero",pos
1,"xxbos * * attention xxmaj spoilers * * \n\n xxmaj first of all , let me say that xxmaj rob xxmaj roy is one of the best films of the 90 's . xxmaj it was an amazing achievement for all those involved , especially the acting of xxmaj liam xxmaj neeson , xxmaj jessica xxmaj lange , xxmaj john xxmaj hurt , xxmaj brian xxmaj cox , and xxmaj tim xxmaj roth . xxmaj michael xxmaj canton xxmaj jones painted a wonderful portrait of the honor and dishonor that men can represent in themselves . xxmaj but alas … \n\n it constantly , and unfairly gets compared to "" braveheart "" . xxmaj these are two entirely different films , probably only similar in the fact that they are both about xxmaj scots in historical xxmaj scotland . xxmaj yet , this comparison frequently bothers me because it seems",pos
2,"xxbos xxmaj the premise of this movie has been tickling my imagination for quite some time now . xxmaj we 've all heard or read about it in some kind of con - text . xxmaj what would you do if you were all alone in the world ? xxmaj what would you do if the entire world suddenly disappeared in front of your eyes ? xxmaj in fact , the last part is actually what happens to xxmaj dave and xxmaj andrew , two room - mates living in a run - down house in the middle of a freeway system . xxmaj andrew is a nervous wreck to say the least and xxmaj dave is considered being one of the biggest losers of society . xxmaj that alone is the main reason to why these two guys get so well along , because they simply only have each",pos
3,"xxbos xxmaj director xxmaj douglas xxmaj sirk once said ` there 's a very short distance between high art and trash , and trash that contains craziness is by this very quality nearer to art ' . xxmaj this statement defines his cinema perfectly , a very unique body of work that includes classic stage adaptations , adventure and war films , westerns and of course , his famous melodramas . \n\n xxmaj sirk 's melodramas were , as the very word signifies , dramas with music . xxmaj the music sets the tone for his masterful style , and every stroke of his brush ( sirk was also a painter ) leaves a powerful image on the screen - turned - canvas . xxmaj but this ai n't life but its representation , an imitation of life . xxmaj sirk never tried to show reality , on the contrary",pos
4,"xxbos xxup myra xxup breckinridge is one of those rare films that established its place in film history immediately . xxmaj praise for the film was absolutely nonexistent , even from the people involved in making it . xxmaj this film was loathed from day one . xxmaj while every now and then one will come across some maverick who will praise the film on philosophical grounds ( aggressive feminism or the courage to tackle the issue of xxunk ) , the film has not developed a cult following like some notorious flops do . xxmaj it 's not hailed as a misunderstood masterpiece like xxup scarface , or trotted out to be ridiculed as a camp classic like xxup showgirls . \n\n xxmaj undoubtedly the reason is that the film , though outrageously awful , is not lovable , or even likable . xxup myra xxup breckinridge is just",neg
5,"xxbos xxmaj it 